### Promp Chaining

In [9]:
from langchain_ollama import ChatOllama

In [10]:
llm = ChatOllama(
    model="granite4:350m",
    #model="llama3-groq-tool-use:8b",
    # model='llama3.2:3b',
    validate_model_on_init=True,
    temperature=0,
)

In [11]:
from typing_extensions import TypedDict

class State(TypedDict):

    topic: str
    joke: str
    improved_joke: str
    final_joke: str

In [12]:
from pydantic import BaseModel, Field

class SearchQuery(BaseModel):
    search_query: str = Field(None, description='Query optimized for search')
    justification: str = Field(None, description="Why this query is relevant to user's request")

In [13]:
structured_llm = llm.with_structured_output(SearchQuery)
output = structured_llm.invoke('How does Caltium CT score relate to cholesterol?')

print(output.search_query)
print(output.justification)

Caltium CT score relates to cholesterol
None


In [7]:
def generate_joke(state: State):
    """
    First LLM call to generate initial joke
    """
    msg = llm.invoke(f"Write a short joke about {state['topic']}")
    return {'joke': msg.content}

def improve_joke(state: State):
    """
    Second LLM call to improve initial joke
    """
    msg = llm.invoke(f"Make this joke funnier by adding wordplay {state['topic']}")
    return {'improved_joke': msg.content}

def polish_joke(state: State):
    """
    Third LLM call to final polish
    """
    msg = llm.invoke(f"Add a surprising twist to this joke {state['topic']}")
    return {'final_joke': msg.content}

def chack_punchline(state: State):
    """
    Gate function to check if joke has a punchline
    """

    if '?' in state["joke"] or '!' in state['joke']:
        return 'Pass'
    
    return 'Final'
